# 🧹 Stage 3 — Data Preprocessing & Cleaning
### B2B Invoice Payment Behaviour Segmentation & Late Payment Predictor
**Course:** Predictive Analytics — Academic Year 2025-26

---

## Stages Covered in This Notebook
- ✅ Stage 3 — Data Preprocessing & Cleaning (complete)

## What This Notebook Does
This notebook handles all raw data preparation tasks:
- Loads and explores the raw dataset
- Cleans and validates data quality
- Creates the target variable
- Detects and treats outliers
- Scales and encodes features
- Splits data into train and test sets
- Handles class imbalance using SMOTE

**Input:**  `Dataset b2b_invoice_payment.csv`  
**Output:** `X_train_preprocessed.csv`, `y_train_preprocessed.csv`, `X_test_preprocessed.csv`, `y_test_preprocessed.csv`

---

In [ ]:
# ── Install dependencies (uncomment if running on Google Colab) ──────────────
# !pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ All libraries loaded successfully.')

---
## Step 1 — Data Loading & Exploration

In [ ]:
df = pd.read_csv('Dataset b2b_invoice_payment.csv')

print(f'Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
df.head(3)

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else '✅ No missing values found.')

In [ ]:
# Numeric summary
print('=== Numeric Summary ===')
df.describe()

In [ ]:
# Categorical columns overview
cat_cols = df.select_dtypes(include='object').columns
print('=== Categorical Columns ===')
for col in cat_cols:
    print(f'{col:35s} → {df[col].nunique()} unique values')

In [ ]:
# Target class distribution
print('=== Target Variable (DelayFlag) ===')
counts = df['DelayFlag'].value_counts()
print(counts)
print(f'\nImbalance ratio: {counts[1]/counts[0]:.2f} : 1  (Late : On-time)')

fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_xticklabels(['On-time (0)', 'Late (1)'], rotation=0)
ax.set_title('Target Variable Distribution')
ax.set_ylabel('Count')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120)
plt.show()

---
## Step 2 — Data Cleaning & Validation

In [ ]:
# ── 2a. Duplicate check ──────────────────────────────────────────────────────
dups = df.duplicated().sum()
print(f'Duplicate rows found: {dups}')
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f'Dropped {dups} duplicates. New shape: {df.shape}')
else:
    print('✅ No duplicates found.')

In [ ]:
# ── 2b. Parse date columns ───────────────────────────────────────────────────
date_cols = ['Doc_Date', 'Net_Due_Date', 'Posting_Date', 'Clearing_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')

print('✅ Date columns parsed. Null count after conversion:')
print(df[date_cols].isnull().sum())

In [ ]:
# ── 2c. Date logic validation ────────────────────────────────────────────────
invalid_date_order = (
    (df['Doc_Date'] > df['Net_Due_Date']) |
    (df['Posting_Date'] > df['Clearing_date'])
).sum()
print(f'Rows with illogical date ordering: {invalid_date_order}')

In [ ]:
# ── 2d. Business range validation ───────────────────────────────────────────
print('Business rule checks:')
print(f'  Amount < 0          : {(df["Amount"] < 0).sum()}')
print(f'  Payment_Term < 0    : {(df["Payment_Term"] < 0).sum()}')
print(f'  Customer Age < 0    : {(df["Age_Of_Customer_Months"] < 0).sum()}')
print('✅ Business rule validation complete.')

In [ ]:
# ── 2e. Fix duplicate column name ────────────────────────────────────────────
df.rename(columns={'Weekday_due.1': 'Weekday_due_num'}, inplace=True)
print('✅ Renamed Weekday_due.1 → Weekday_due_num')

---
## Step 3 — Target Variable Creation

`DelayFlag` already exists in the dataset:
- `1` = invoice paid **late** (Days_Overdue_Delay > 0)
- `0` = paid **on time** or early

We verify consistency and rename for clarity.

In [ ]:
# Verify consistency
mismatch = ((df['DelayFlag'] == 0) & (df['Days_Overdue_Delay'] > 0)).sum()
print(f'Mismatch (Flag=0 but Delay>0): {mismatch}')

# Rename for clarity
df.rename(columns={'DelayFlag': 'target_late_payment'}, inplace=True)

print('\n✅ Target variable created:')
print(df['target_late_payment'].value_counts().rename({0: 'On-time', 1: 'Late'}))

---
## Step 4 — Outlier Detection & Treatment

> **Note:** Feature Engineering (creating new columns) is handled separately in `stage5_feature_engineering.ipynb`. Here we only treat outliers on the existing raw columns.

In [ ]:
# ── 4a. Visualise outliers ───────────────────────────────────────────────────
outlier_cols = ['Amount', 'Payment_Term', 'Age_Of_Customer_Year']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(14, 4))
for ax, col in zip(axes, outlier_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
    ax.set_ylabel('Value')
plt.suptitle('Boxplots — Outlier Visualisation', y=1.02)
plt.tight_layout()
plt.savefig('outlier_boxplots.png', dpi=120)
plt.show()

In [ ]:
# ── 4b. Winsorization — cap outliers at 1st and 99th percentile ──────────────
# Capping is preferred over dropping to preserve all customer invoice records.

def iqr_cap(series, lower_pct=0.01, upper_pct=0.99):
    lo = series.quantile(lower_pct)
    hi = series.quantile(upper_pct)
    return series.clip(lo, hi), lo, hi

cap_cols = ['Amount', 'Payment_Term']
cap_log = {}
for col in cap_cols:
    df[col], lo, hi = iqr_cap(df[col])
    cap_log[col] = (round(lo, 2), round(hi, 2))

print('✅ Winsorization complete. Bounds applied:')
for k, v in cap_log.items():
    print(f'  {k:20s}: lower={v[0]:>10,.2f}  upper={v[1]:>10,.2f}')

---
## Step 5 — Drop Leakage & Redundant Columns

These columns must be removed **before** any modelling.

### Leakage Columns (known only AFTER payment is made)
| Column | Reason |
|---|---|
| `Days_Overdue_Delay` | Directly encodes the target |
| `Delay_Bins` | Binned version of above |
| `Clearing_date` | Only known after clearing |
| `Clearing_doc` | Assigned at clearing time |
| `Weekday_clearing` | Day of clearing event |
| `Weekday_clearnum` | Numeric version of above |
| `Quarter_clearing` | Quarter of clearing event |

In [ ]:
LEAKAGE_COLS = [
    'Days_Overdue_Delay', 'Delay_Bins', 'Clearing_date',
    'Clearing_doc', 'Weekday_clearing', 'Weekday_clearnum', 'Quarter_clearing'
]

REDUNDANT_COLS = [
    'Document_No', 'Cust_Num', 'Customer_Name',
    'Age_Of_Customer_Months',           # redundant with Age_Of_Customer_Year
    'Amount_Bins', 'Payment_Term_Bins', 'Customer_Age_Year_Bins',  # binned duplicates
    'Weekday_due',                       # string duplicate of Weekday_due_num
    'Doc_Date', 'Net_Due_Date', 'Posting_Date'  # raw dates — features extracted in Stage 5
]

DROP_COLS = list(set(LEAKAGE_COLS + REDUNDANT_COLS))
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print(f'Columns before drop : {df.shape[1]}')
print(f'Columns after drop  : {df_clean.shape[1]}')
print(f'\n✅ Remaining columns:')
print(df_clean.columns.tolist())

---
## Step 6 — Encoding & Scaling

In [ ]:
# ── 6a. Label encode categorical columns ─────────────────────────────────────
cat_features = df_clean.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_features)

le_dict = {}
for col in cat_features:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    le_dict[col] = le
    print(f'  ✅ {col:35s} → {len(le.classes_)} classes encoded')

In [ ]:
# ── 6b. Separate X and y ─────────────────────────────────────────────────────
X = df_clean.drop(columns=['target_late_payment'])
y = df_clean['target_late_payment']

print(f'Feature matrix X : {X.shape}')
print(f'Target vector  y : {y.shape}')

---
## Step 7 — Stratified Train-Test Split

> ⚠️ **Important:** Split happens BEFORE scaling. The scaler is fit on training data only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Training set : {X_train.shape[0]:,} rows  |  Late={y_train.sum():,}  On-time={(y_train==0).sum():,}')
print(f'Test set     : {X_test.shape[0]:,} rows  |  Late={y_test.sum():,}  On-time={(y_test==0).sum():,}')

In [ ]:
# ── Fit scaler on TRAIN only — then apply to TEST ────────────────────────────
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X.columns
)

print('✅ Scaling complete.')
print('Train mean (should be ~0):', X_train_scaled.mean().mean().round(6))
print('Train std  (should be ~1):', X_train_scaled.std().mean().round(6))

---
## Step 8 — Class Imbalance Handling (SMOTE)

> ⚠️ **SMOTE is applied on the training set ONLY. The test set is never modified.**

In [ ]:
print('Before SMOTE:')
print(y_train.value_counts().rename({0: 'On-time', 1: 'Late'}))

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_train_res).value_counts().rename({0: 'On-time', 1: 'Late'}))
print(f'\n✅ Resampled training shape: {X_train_res.shape}')

In [ ]:
# Visualise before / after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(['On-time', 'Late'], y_train.value_counts().sort_index(),
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Before SMOTE')
axes[1].bar(['On-time', 'Late'], pd.Series(y_train_res).value_counts().sort_index(),
            color=['steelblue', 'tomato'], edgecolor='black')
axes[1].set_title('After SMOTE')
for ax in axes:
    ax.set_ylabel('Count')
plt.suptitle('Class Balance — Before vs After SMOTE')
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=120)
plt.show()

---
## Step 9 — Data Leakage Prevention (Final Audit)

In [ ]:
leakage_audit = {
    'Clearing_date removed'        : 'Clearing_date' not in X.columns,
    'Days_Overdue_Delay removed'   : 'Days_Overdue_Delay' not in X.columns,
    'Delay_Bins removed'           : 'Delay_Bins' not in X.columns,
    'Clearing_doc removed'         : 'Clearing_doc' not in X.columns,
    'Weekday_clearing removed'     : 'Weekday_clearing' not in X.columns,
    'Scaler fit on train-only'     : True,
    'SMOTE applied on train-only'  : True,
    'Test set untouched by SMOTE'  : True,
}

print('=== Data Leakage Prevention Audit ===')
all_clear = True
for check, passed in leakage_audit.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    if not passed: all_clear = False
    print(f'  {status}  |  {check}')
print()
print('✅ All leakage checks passed!' if all_clear else '⚠️ Some checks FAILED.')

---
## Step 10 — Save Outputs

In [ ]:
X_train_res_df = pd.DataFrame(X_train_res, columns=X.columns)
y_train_res_s  = pd.Series(y_train_res, name='target_late_payment')

X_train_res_df.to_csv('X_train_preprocessed.csv', index=False)
y_train_res_s.to_csv('y_train_preprocessed.csv',  index=False)
X_test_scaled.to_csv('X_test_preprocessed.csv',   index=False)
y_test.to_csv('y_test_preprocessed.csv',           index=False)

print('✅ All output files saved:')
print(f'   X_train_preprocessed.csv  →  {X_train_res_df.shape[0]:,} rows × {X_train_res_df.shape[1]} cols')
print(f'   y_train_preprocessed.csv  →  {len(y_train_res_s):,} rows')
print(f'   X_test_preprocessed.csv   →  {X_test_scaled.shape[0]:,} rows × {X_test_scaled.shape[1]} cols')
print(f'   y_test_preprocessed.csv   →  {len(y_test):,} rows')
print()
print('➡️  Ready for Stage 5: Feature Engineering')